# Supply Chain Routing: MILP vs Q-Learning Comparison

In [ ]:
from pathlib import Path
from rl.train import run_all

if not Path("results/milp_global.csv").exists():
    run_all(episodes=15_000, num_milp_simulations=10)
else:
    print("Results already exist — loading from CSVs.")

## Cost Comparison Table

In [ ]:
import pandas as pd

milp_global = pd.read_csv("results/milp_global.csv")
milp_daily = pd.read_csv("results/milp_daily.csv")
rl = pd.read_csv("results/rl_policy.csv")

milp_opt = milp_global["total_cost"].iloc[0]
daily_mean = milp_daily["total_cost"].mean()
daily_std = milp_daily["total_cost"].std()
rl_mean = rl["total_cost"].mean()
rl_std = rl["total_cost"].std()

summary = pd.DataFrame([
    {"Method": "MILP Global (optimal)", "Mean Cost ($)": f"{milp_opt:,.0f}", "Std ($)": "--", "Gap vs Optimal": "--"},
    {"Method": "MILP Daily Myopic",     "Mean Cost ($)": f"{daily_mean:,.0f}", "Std ($)": f"{daily_std:,.0f}", "Gap vs Optimal": f"{(daily_mean/milp_opt - 1)*100:.1f}%"},
    {"Method": "Q-Learning",            "Mean Cost ($)": f"{rl_mean:,.0f}",    "Std ($)": f"{rl_std:,.0f}",    "Gap vs Optimal": f"{(rl_mean/milp_opt - 1)*100:.1f}%"},
])
summary.set_index("Method")

## Learning Curve

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

lc = pd.read_csv("results/learning_curve.csv")
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lc["episode"], lc["episode_reward"], alpha=0.2, color="steelblue", label="Episode reward")
ax.plot(lc["episode"], lc["smoothed_reward"], color="steelblue", linewidth=2, label="Smoothed (100-ep avg)")
ax.set_xlabel("Episode")
ax.set_ylabel("Cumulative reward (negative cost)")
ax.set_title("Q-Learning Training Curve")
ax.legend()
plt.tight_layout()
plt.show()

## Learned Policy Table

In [ ]:
import pandas as pd

policy = pd.read_csv("results/rl_policy_table.csv")
policy["demand_label"] = policy["demand_bucket"].map({0: "Low", 1: "Med", 2: "High"})
pivot = policy.pivot(index="day", columns="demand_label", values="open_dcs")
print("Learned DC policy: open_dcs per (day, demand level)")
pivot